In this notebook we try to connect comfy UI workflows to the project

imports

In [8]:
from pathlib import Path
import os
from ObjaverseDataset import ObjaverseDataset
from torch.utils.data import DataLoader
from pathlib import Path

from prompts import *
from utils import _convert_trimesh_to_o3d

# Keep relative input and output paths anchored at the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "experiments" else Path.cwd()
if not (PROJECT_ROOT / "utils.py").is_file():
    raise RuntimeError(f"Open this notebook from the project root or experiments/: {Path.cwd()}")
os.chdir(PROJECT_ROOT)

from utils import *

In [3]:
import requests
import json
import time
from pathlib import Path

import trimesh
from IPython.display import Image, display
import random

import utils
from prompts import OBJECT_PART_GENERATION_PROMPT, PROMPT_ENHANCEMENT_SYSTEM_PROMPT

COMFY_URL = "http://192.168.178.23:8188"
response = requests.get(f"{COMFY_URL}/system_stats")

print(response.status_code)

200


The generation of 3d objects consists of two parts: genertate image from prompt which is provided from VLM-agent
Then this image is used to generate 3d object

1) Text to image

In [4]:
with open("../comfyUI_workflows/image_flux2_text_to_image_9b.json") as f:
    workflow = json.load(f)
    for node_id, node in workflow.items():
        print(node_id, node["class_type"])

9 SaveImage
75:61 KSamplerSelect
75:62 Flux2Scheduler
75:63 CFGGuider
75:64 SamplerCustomAdvanced
75:65 VAEDecode
75:66 EmptyFlux2LatentImage
75:67 CLIPTextEncode
75:68 PrimitiveInt
75:69 PrimitiveInt
75:73 RandomNoise
75:70 UNETLoader
75:71 CLIPLoader
75:72 VAELoader
75:74 CLIPTextEncode


In [4]:
with open("../comfyUI_workflows/image_flux2_text_to_image_9b.json") as f:
    workflow = json.load(f)
    workflow["75:74"]["inputs"]["text"] = "A wooden chair"
    workflow["75:73"]["inputs"]["noise_seed"] = random.randint(0, 2**32 - 1)
    response = requests.post(
    f"{COMFY_URL}/prompt",
    json={"prompt": workflow}
)

response.raise_for_status()
prompt_id = response.json()["prompt_id"]

print(prompt_id)

f381b41f-60aa-4806-b59f-24102e2bc68b


In [5]:
def wait_for_completion(prompt_id, timeout=1000, poll_interval=1.0):
    """

    :param prompt_id: id of image in queue
    :param timeout: time to wait for completion
    :param poll_interval: time between requests
    :return:
    """
    start = time.time()

    while time.time() - start < timeout:
        r = requests.get(f"{COMFY_URL}/history/{prompt_id}")
        r.raise_for_status()
        history = r.json()

        # When image is generated, it appears in queue
        if prompt_id in history:
            item = history[prompt_id]

            status = item.get("status", {})
            if status.get("status_str") == "error":
                raise RuntimeError(f"ComfyUI error: {status}")

            return item

        time.sleep(poll_interval)

    raise TimeoutError(f"Prompt {prompt_id} did not finish in {timeout} seconds")


def extract_image_info(history_item, save_node_id="9"):
    """
    save_node_id = ID of SaveImage node in comfy config.
    9 for current workflow
    """
    outputs = history_item.get("outputs", {})
    if save_node_id not in outputs:
        raise KeyError(f"Node {save_node_id} not found in outputs: {list(outputs.keys())}")

    node_output = outputs[save_node_id]
    images = node_output.get("images", [])

    if not images:
        raise ValueError("No images returned by SaveImage node")

    return images[0]  # take first image


def download_file(file_info, output_path="generated.png"):
    """
    image_info looks like that:
    {
        'filename': 'Flux2-Klein_00001_.png',
        'subfolder': '',
        'type': 'output'
    }
    """
    r = requests.get(f"{COMFY_URL}/view", params=file_info)
    r.raise_for_status()

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_bytes(r.content)

    return output_path

In [ ]:
history_item = wait_for_completion(prompt_id)
image_info = extract_image_info(history_item, save_node_id="9")
image_path = download_file(image_info, output_path="../generated_images/chair.png")

print("Saved to:", image_path)
display(Image(filename=str(image_path)))

In [7]:
def generate_image_from_text(prompt):
    """
    Generates image from comfy UI for given text prompt
    :param prompt: text describing the image that should be generated
    :return: path to generated image
    """
    COMFY_URL = "http://192.168.178.23:8188"
    with open("../comfyUI_workflows/image_flux2_text_to_image_9b.json") as f:
        workflow = json.load(f)
        workflow["75:74"]["inputs"]["text"] = OBJECT_PART_GENERATION_PROMPT + prompt
        workflow["75:73"]["inputs"]["noise_seed"] = random.randint(0, 2**32 - 1) # for random
    response = requests.post(
    f"{COMFY_URL}/prompt",
    json={"prompt": workflow}
    )

    response.raise_for_status()
    prompt_id = response.json()["prompt_id"]
    history_item = wait_for_completion(prompt_id)
    image_info = extract_image_info(history_item, save_node_id="9")
    image_path = download_file(image_info, output_path="../generated_images/prompt.png")
    display(Image(filename=str(image_path)))
    return image_path

In [ ]:
generate_image_from_text("capybara")

2) Image to 3d object

In [9]:
with open("../comfyUI_workflows/pixal3d_trellis2_image_to_model.json") as f:
    workflow = json.load(f)
    for node_id, node in workflow.items():
        print(node_id, node["class_type"])

3 KSampler
4 VoxelToMesh
12 KSampler
15 CLIPVisionLoader
18 KSampler
23 KSampler
40 UNETLoader
55 LoadMoGeModel
56 MoGeInference
87 EmptyTrellis2LatentStructure
91 Trellis2ShapeStage
92 VaeDecodeShapeTrellis
93 VaeDecodeTextureTrellis
94 Trellis2UpsampleStage
98 Trellis2TextureStage
108 ModelSamplingSD3
117 VAELoader
118 VAELoader
119 VaeDecodeStructureTrellis2
122 LoadImage
125 RescaleCFG
126 RescaleCFG
147 BakeTextureFromVoxel
164 PreviewImage
186 DecimateMesh
192 RemoveBackground
193 LoadBackgroundRemovalModel
196 UnwrapMesh
199 CFGOverride
202 GetMeshInfo
207 PreviewImage
208 PreviewImage
210 ApplyTextureToMesh
224 BakeNormalMapFromMesh
226 PreviewImage
233 BakeAmbientOcclusion
235 PreviewImage
238 MeshSmoothNormals
241 RemeshMesh
242 MoGeGeometryToFOV
246 Preview3DAdvanced
247 MeshToFile3D
248 ComfySwitchNode
252 PaintMesh
260 MeshSmoothNormals
261 RenderUVAtlas
262 PreviewImage
279 CFGOverride
282 MeshToFile3D
285 MeshToFile3D
288 PrimitiveInt
298 Pixal3DConditioning
299 Trellis2

First, upload the image to the server

In [10]:
def upload_image(image_path):
    image_path = Path(image_path)

    with open(image_path, "rb") as f:
        response = requests.post(
            f"{COMFY_URL}/upload/image",
            files={"image": (image_path.name, f)},
            data={"type": "input", "overwrite": "false"}
        )

    response.raise_for_status()

    return response.json()

In [11]:
image_info = upload_image("../generated_images/prompt.png")
print(image_info)

{'name': 'prompt (5).png', 'subfolder': '', 'type': 'input', 'asset': {'id': '535098fd-0e4c-4955-bea1-9efbd15fcd1d', 'name': 'prompt (5).png', 'asset_hash': 'blake3:8a69daacae14c8445accc8c989bad9a288ade105bcc8aedc584502cad8e869d3', 'size': 969407, 'mime_type': 'image/png', 'tags': ['input', 'uploaded'], 'preview_url': '/api/view?type=input&filename=prompt%20%285%29.png'}}


In [12]:
with open("../comfyUI_workflows/pixal3d_trellis2_image_to_model.json") as f:
    workflow = json.load(f)

workflow["122"]["inputs"]["image"] = image_info["name"]
response = requests.post(
    f"{COMFY_URL}/prompt",
    json={"prompt": workflow}
)

response.raise_for_status()

prompt_id = response.json()["prompt_id"]

history_item = wait_for_completion(prompt_id)

In [13]:
model_info = history_item["outputs"]["322"]["3d"][0]

model_path = download_file(
    model_info,
    "../generated_models/" + model_info["filename"]
)


In [ ]:
model_path = Path("../generated_models/" + model_info["filename"])
mesh = trimesh.load(model_path)
mesh.show()

In [15]:
def generate_3d_object_from_image(image_path):
    """
    Generates 3d object from image
    :param image_path: path to image
    :return: path to generated 3d object
    """
    image_info = upload_image(image_path)
    with open("../comfyUI_workflows/pixal3d_trellis2_image_to_model.json") as f:
        workflow = json.load(f)

    print("Input image:", image_path)
    print("Uploaded image:", image_info)

    workflow["122"]["inputs"]["image"] = image_info["name"]
    response = requests.post(
        f"{COMFY_URL}/prompt",
        json={"prompt": workflow}
    )
    response.raise_for_status()
    prompt_id = response.json()["prompt_id"]
    history_item = wait_for_completion(prompt_id)
    model_info = history_item["outputs"]["322"]["3d"][0]
    model_path = download_file(
        model_info,
        "../generated_models/" + model_info["filename"]
    )
    return model_path


In [ ]:
mesh = trimesh.load(generate_3d_object_from_image("../generated_images/prompt.png"))
mesh.show()

Now the whole pipeline together

In [ ]:
prompt = "A close-up studio macro photograph of a single isolated yellow-orange duck beak floating in the center, completely detached, no duck, no head, no feathers, no body, solid pure pitch black background, dramatic studio lighting, sharp focus, ultra-detailed texture, 8k resolution --no duck, head, eyes, feathers, body"
image_path = generate_image_from_text(prompt)
mesh_path = generate_3d_object_from_image(image_path)
mesh = trimesh.load(mesh_path)
mesh.show()

In [ ]:
prompt = "Duck beak"
image_path = utils.generate_image_from_text(prompt)
mesh_path = utils.generate_3d_object_from_image(image_path)
mesh = trimesh.load(mesh_path)
mesh.show()

In [6]:
import ollama


def enhance_prompt(prompt, ollama_host):
    client = ollama.Client(host=ollama_host)

    response = client.chat(
        model="qwen3-vl:8b",
        messages=[
            {
                "role": "system",
                "content": PROMPT_ENHANCEMENT_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        think=False,
        options={"temperature": 0.2}
    )
    #print(response.model_dump_json(indent=2))

    return response["message"]["content"]

In [ ]:
prompt = "Car wheel"
# enhanced_prompt = enhance_prompt(prompt, "192.168.178.23")
# print("\n\n\n\n")
# print(enhanced_prompt)
# enhanced_prompt
image_path = utils.generate_image_from_text(prompt)
mesh_path = utils.generate_3d_object_from_image(image_path)
mesh = trimesh.load(mesh_path)
mesh.show()